# Módulo 17 · Práctica: el roofline de tu CPU y el coste de entrenar un LLM

En este notebook mides cuántas operaciones por segundo hace tu ordenador al multiplicar matrices, lo explicas con el **modelo roofline** y construyes una **calculadora de coste, tiempo y energía** de entrenamiento con $C \approx 6ND$. Son las mismas cuentas que hace un equipo antes de lanzar el entrenamiento de un LLM.

**Cómo trabajar:** ejecuta las celdas en orden. Las celdas `# TU CÓDIGO` son ejercicios: complétalas y ejecuta la celda `# Comprobación` que va detrás. Las cifras de GPU son aproximadas (A100 SXM 80 GB: ~312 TFLOP/s bf16 densos, ~2 TB/s, ~400 W).

| Parte | Qué haces |
| --- | --- |
| 1 | Contar FLOP y medir el rendimiento de `A @ B` según el tamaño |
| 2 | Medir el ancho de banda y la intensidad aritmética |
| 3 | Construir el roofline de tu CPU y de una A100 |
| 4 | Calculadora de coste, tiempo y energía de entrenamiento |
| 5 | Memoria, ZeRO y burbuja del pipeline |
| 6 | Física: Landauer frente a la energía real |
| 7 | Retos |

## Parte 1 · Contar FLOP y medir `A @ B`

Un producto $A_{m\times k} B_{k\times n}$ cuesta $2mkn$ FLOP. Para medir bien hay que calentar (la primera llamada es más lenta), repetir y quedarse con el mejor tiempo.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

def mejor_tiempo(f, repeticiones=5):
    # Devuelve el menor tiempo (s) de varias ejecuciones de f(), tras una de calentamiento.
    f()
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        f()
        tiempos.append(time.perf_counter() - t0)
    return min(tiempos)

**Ejercicio 1.** Escribe `flops_matmul(m, k, n)` que devuelva los FLOP de multiplicar una matriz $m\times k$ por otra $k\times n$.

In [ ]:
# TU CÓDIGO
def flops_matmul(m, k, n):
    return None

In [ ]:
# Comprobación
assert flops_matmul(2, 3, 4) == 48, "Un producto (2x3)(3x4) cuesta 2·2·3·4 = 48 FLOP"
assert flops_matmul(2048, 4096, 4096) == 2 * 2048 * 4096 * 4096, "Revisa la fórmula 2mkn"
print("Correcto")

**Ejercicio 2.** Completa `gflops_matmul(n)`: crea dos matrices aleatorias $n\times n$ en `float32`, mide `A @ B` con `mejor_tiempo` y devuelve los GFLOP/s ($10^9$ FLOP/s). Para tamaños pequeños, repite el producto `r` veces dentro de la función medida.

In [ ]:
# TU CÓDIGO
def gflops_matmul(n):
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    r = max(1, int(5e7 / n**3))      # repeticiones internas para que el tiempo sea medible
    def f():
        for _ in range(r):
            A @ B
    t = None                          # mide f con mejor_tiempo
    return None                       # GFLOP/s

In [ ]:
# Comprobación
tamanos = [16, 32, 64, 128, 256, 512, 1024]
gf = [gflops_matmul(n) for n in tamanos]
for n, g in zip(tamanos, gf):
    print(f"n = {n:5d}: {g:8.1f} GFLOP/s")
assert all(g is not None and g > 0 for g in gf), "gflops_matmul debe devolver un número positivo"
assert max(gf[3:]) > 3 * gf[0], "Las matrices grandes deberían rendir bastante más que las de 16x16"
print("Correcto: las matrices grandes aprovechan mucho mejor la CPU")

In [ ]:
plt.semilogx(tamanos, gf, "o-", base=2)
plt.xlabel("n (matrices n x n)"); plt.ylabel("GFLOP/s"); plt.title("Rendimiento de A @ B en tu CPU")
plt.grid(True, alpha=0.3); plt.show()

Con $n$ pequeño dominan el coste fijo y la memoria; al crecer $n$, cada número se reutiliza $n$ veces. Si cae para $n$ grande, las matrices ya no caben en la caché.

## Parte 2 · Ancho de banda e intensidad aritmética

Una operación elemento a elemento como `y = 2 * x` hace 1 FLOP por cada 8 bytes (lee 4 y escribe 4 en `float32`). Su velocidad mide, en la práctica, el **ancho de banda** de la memoria.

**Ejercicio 3.** Mide el ancho de banda en GB/s: bytes movidos (lectura de `x` + escritura de `y`) divididos por el tiempo de `np.multiply(x, 2.0, out=y)`.

In [ ]:
# TU CÓDIGO
x = rng.random(20_000_000, dtype=np.float32)
y = np.empty_like(x)
t = None
ancho_banda = None    # GB/s

In [ ]:
# Comprobación
assert ancho_banda is not None and 0.1 < ancho_banda < 2000, "El ancho de banda debe estar en GB/s (entre 0,1 y 2000)"
print(f"Ancho de banda medido: {ancho_banda:.1f} GB/s")
print("Correcto")

**Ejercicio 4.** Escribe `intensidad(m, k, n, bytes_por_numero)`: FLOP divididos por los bytes mínimos movidos, suponiendo que se leen $A$ y $B$ y se escribe $C$ una sola vez:

$$I = \frac{2mkn}{b\,(mk + kn + mn)}$$

In [ ]:
# TU CÓDIGO
def intensidad(m, k, n, bytes_por_numero):
    return None

In [ ]:
# Comprobación
assert np.isclose(intensidad(1024, 1024, 1024, 2), 1024 / 3), "Una matmul cuadrada en bf16 tiene I = n/3"
assert np.isclose(intensidad(4096, 4096, 1, 2), 0.99951, atol=1e-4), "Matriz por vector: I ≈ 1"
print("Correcto")

## Parte 3 · El modelo roofline

$$P_{\text{alcanzable}}(I) = \min\left(P_{\text{pico}},\; I \cdot B_{\text{mem}}\right), \qquad I^* = \frac{P_{\text{pico}}}{B_{\text{mem}}}$$

**Ejercicio 5.** Escribe `roofline(I, pico, ancho)` (mismas unidades: FLOP/s y bytes/s) y calcula la intensidad crítica de la A100.

In [ ]:
# TU CÓDIGO
A100_PICO = 312e12     # FLOP/s bf16 densos (aprox.)
A100_BW = 2e12         # bytes/s de HBM (aprox.)

def roofline(I, pico, ancho):
    return None

I_critica_a100 = None

In [ ]:
# Comprobación
assert np.isclose(I_critica_a100, 156), "I* = 312e12 / 2e12 = 156 FLOP/byte"
assert np.isclose(roofline(21.33, A100_PICO, A100_BW), 42.66e12), "Con I = 21,33 la A100 da ~42,7 TFLOP/s"
assert np.isclose(roofline(1000, A100_PICO, A100_BW), A100_PICO), "Por encima de I* manda el pico"
print("Correcto")

Ahora el roofline de **tu CPU**. Tomamos como pico estimado el mejor rendimiento que mediste y como ancho de banda el de la parte 2. Los puntos son tus medidas de matmul, cada una con su intensidad en `float32` (4 bytes).

In [ ]:
pico_cpu = max(gf) * 1e9
bw_cpu = ancho_banda * 1e9
Is = np.logspace(-1, 3, 200)

plt.loglog(Is, roofline(Is, pico_cpu, bw_cpu) / 1e9, label="roofline estimado de tu CPU")
Ix = [intensidad(n, n, n, 4) for n in tamanos]
plt.loglog(Ix, gf, "o", label="matmul medidas")
plt.loglog(Is, roofline(Is, A100_PICO, A100_BW) / 1e9, "--", label="A100 bf16 (aprox.)")
plt.xlabel("Intensidad aritmética (FLOP/byte)"); plt.ylabel("GFLOP/s")
plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()
print(f"Intensidad crítica de tu CPU ≈ {pico_cpu / bw_cpu:.1f} FLOP/byte")

**Ejercicio 6.** Mide los GFLOP/s de una matriz por vector (`A @ v`, $n = 2048$, `float32`) y comprueba que está muy por debajo de la matmul cuadrada. Calcula también su intensidad.

In [ ]:
# TU CÓDIGO
n = 2048
A = rng.random((n, n), dtype=np.float32)
v = rng.random(n, dtype=np.float32)
gf_matvec = None
I_matvec = None

In [ ]:
# Comprobación
assert I_matvec is not None and I_matvec < 0.6, "En float32, matriz por vector tiene I ≈ 0,5"
assert gf_matvec < max(gf), "La matriz por vector debería rendir menos que la mejor matmul"
print(f"matvec: {gf_matvec:.2f} GFLOP/s con I = {I_matvec:.2f}; mejor matmul: {max(gf):.1f} GFLOP/s")
print("Correcto: así se siente generar tokens de uno en uno (módulo 19)")

## Parte 4 · Calculadora de coste, tiempo y energía

$$C = 6ND, \qquad t = \frac{C}{P_{\text{pico}}\cdot \text{MFU}\cdot n_{\text{GPU}}}, \qquad E = \text{GPU·h}\cdot W \cdot \text{PUE}$$

**Ejercicio 7.** Completa `coste_entrenamiento`. Devuelve un diccionario con `flop`, `gpu_horas`, `dias`, `energia_mwh` y `dolares`.

In [ ]:
# TU CÓDIGO
def coste_entrenamiento(N, D, pico=312e12, mfu=0.4, n_gpu=1024, vatios=400, pue=1.2, precio_gpu_hora=2.0):
    C = None
    gpu_horas = None
    dias = None
    energia_mwh = None
    dolares = None
    return dict(flop=C, gpu_horas=gpu_horas, dias=dias, energia_mwh=energia_mwh, dolares=dolares)

In [ ]:
# Comprobación (modelo tipo Chinchilla en A100 hipotéticas)
r = coste_entrenamiento(70e9, 1.4e12)
assert np.isclose(r["flop"], 5.88e23), "C = 6·70e9·1,4e12"
assert np.isclose(r["gpu_horas"], 1.3088e6, rtol=1e-3), "GPU-horas ≈ 1,31 millones"
assert np.isclose(r["dias"], 53.25, rtol=1e-3), "Con 1.024 GPU, unos 53 días"
assert np.isclose(r["energia_mwh"], 628.2, rtol=1e-3), "Energía ≈ 628 MWh"
assert np.isclose(r["dolares"], 2.6175e6, rtol=1e-3), "Coste ≈ 2,6 millones de dólares"
for k, v in r.items():
    print(f"{k:12s} {v:.4g}")
print("Correcto")

**Ejercicio 8.** Usa la calculadora: un modelo de $10^9$ parámetros con $2\cdot 10^{10}$ tokens, MFU = 0,5, 8 GPU y PUE = 1. ¿Cuántos días y cuántas GPU-horas?

In [ ]:
# TU CÓDIGO
r8 = None

In [ ]:
# Comprobación
assert np.isclose(r8["flop"], 1.2e20), "C = 6·1e9·2e10 = 1,2e20"
assert np.isclose(r8["gpu_horas"], 213.68, rtol=1e-3), "Unas 214 GPU-horas"
assert np.isclose(r8["dias"], 1.113, rtol=1e-3), "Algo más de un día con 8 GPU"
print(f"{r8['dias']:.2f} días, {r8['gpu_horas']:.0f} GPU-horas, {r8['dolares']:.0f} $")
print("Correcto")

**Ejercicio 9.** Con un presupuesto de 100.000 GPU-horas (A100, MFU = 0,4) y la regla de Chinchilla $D = 20N$, halla el $N$ y el $D$ óptimos. Pista: $C = 120N^2$.

In [ ]:
# TU CÓDIGO
C_presupuesto = None
N_opt = None
D_opt = None

In [ ]:
# Comprobación
assert np.isclose(C_presupuesto, 4.4928e22), "C = GPU-h · 3600 · pico · MFU"
assert np.isclose(N_opt, 1.935e10, rtol=1e-3), "N ≈ 19.300 millones de parámetros"
assert np.isclose(D_opt, 3.87e11, rtol=1e-3), "D ≈ 387.000 millones de tokens"
print(f"N ≈ {N_opt:.3g}, D ≈ {D_opt:.3g}")
print("Correcto")

Explora cómo cambia el tiempo con la MFU: pasar del 30 % al 60 % divide por dos la factura.

In [ ]:
mfus = np.linspace(0.2, 0.7, 6)
dias = [coste_entrenamiento(70e9, 1.4e12, mfu=m)["dias"] for m in mfus]
for m, d in zip(mfus, dias):
    print(f"MFU = {m:.0%}: {d:6.1f} días")

## Parte 5 · Memoria, ZeRO y burbuja

**Ejercicio 10.** Escribe `memoria_zero(psi, n_gpu, etapa)` que devuelva los bytes por GPU con Adam y precisión mixta (2 + 2 + 12 bytes por parámetro): etapa 0 → $16\Psi$; 1 → $4\Psi + 12\Psi/N$; 2 → $2\Psi + 14\Psi/N$; 3 → $16\Psi/N$. Escribe también `burbuja(p, m)` $= (p-1)/(m+p-1)$.

In [ ]:
# TU CÓDIGO
def memoria_zero(psi, n_gpu, etapa):
    return None

def burbuja(p, m):
    return None

In [ ]:
# Comprobación
gb = [memoria_zero(7.5e9, 64, e) / 1e9 for e in range(4)]
assert np.allclose(gb, [120, 31.40625, 16.640625, 1.875]), "Tabla de ZeRO con Ψ = 7.500 M y N = 64"
assert np.isclose(burbuja(4, 8), 3 / 11) and np.isclose(burbuja(4, 32), 3 / 35), "Burbuja = (p-1)/(m+p-1)"
print("ZeRO (GB por GPU):", [round(g, 2) for g in gb])
print("Correcto")

## Parte 6 · Física: Landauer frente a la realidad

$$E_{\min} = k_B T \ln 2$$

**Ejercicio 11.** Calcula el límite de Landauer a 300 K (en julios), la energía por FLOP de una A100 (400 W / 312 TFLOP/s) y la razón entre ambas.

In [ ]:
# TU CÓDIGO
k_B = 1.380649e-23   # J/K (valor exacto en el SI)
E_landauer = None
E_flop_a100 = None
razon = None

In [ ]:
# Comprobación
assert np.isclose(E_landauer, 2.871e-21, rtol=1e-3), "kT ln 2 a 300 K ≈ 2,87e-21 J"
assert np.isclose(E_flop_a100, 1.282e-12, rtol=1e-3), "≈ 1,28 pJ por FLOP"
assert 4e8 < razon < 5e8, "La A100 está unas 4,5e8 veces por encima del límite"
print(f"Landauer: {E_landauer:.3e} J; A100: {E_flop_a100:.3e} J/FLOP; razón {razon:.2e}")
print("Correcto")

In [ ]:
# ¿Cuánta agua hace falta para enfriar 1.024 GPU (409,6 kW) si el agua se calienta 10 K?
potencia = 1024 * 400            # W
caudal = potencia / (4186 * 10)  # kg/s
print(f"{potencia/1e3:.1f} kW -> {caudal:.1f} kg/s de agua")

## Parte 7 · Retos

1. **Tu MFU.** Toma tu mejor GFLOP/s y compáralo con el pico teórico de tu CPU (núcleos × frecuencia × FLOP por ciclo; busca tu modelo con `lscpu`). ¿Qué MFU obtienes?
2. **Lote en inferencia.** Calcula `intensidad(B, 4096, 4096, 2)` para B = 1, 16, 64, 256. ¿A partir de qué lote se vuelve limitada por cómputo en la A100?
3. **Softmax online.** Implementa la softmax por bloques de la sección 17.6 y comprueba que coincide con la normal para un vector aleatorio de 1.000 elementos.

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
# def flops_matmul(m, k, n): return 2 * m * k * n
# Ejercicio 2
#     t = mejor_tiempo(f);  return r * flops_matmul(n, n, n) / t / 1e9
# Ejercicio 3
# t = mejor_tiempo(lambda: np.multiply(x, 2.0, out=y));  ancho_banda = 2 * x.nbytes / t / 1e9
# Ejercicio 4
# return flops_matmul(m, k, n) / (bytes_por_numero * (m*k + k*n + m*n))
# Ejercicio 5
# def roofline(I, pico, ancho): return np.minimum(pico, np.asarray(I) * ancho)
# I_critica_a100 = A100_PICO / A100_BW
# Ejercicio 6
# gf_matvec = flops_matmul(n, n, 1) / mejor_tiempo(lambda: A @ v) / 1e9;  I_matvec = intensidad(n, n, 1, 4)
# Ejercicio 7
# C = 6*N*D;  gpu_horas = C / (pico*mfu) / 3600;  dias = gpu_horas / n_gpu / 24
# energia_mwh = gpu_horas * vatios * pue / 1e6;  dolares = gpu_horas * precio_gpu_hora
# Ejercicio 8: r8 = coste_entrenamiento(1e9, 2e10, mfu=0.5, n_gpu=8, pue=1.0)
# Ejercicio 9
# C_presupuesto = 1e5 * 3600 * 312e12 * 0.4;  N_opt = np.sqrt(C_presupuesto / 120);  D_opt = 20 * N_opt
# Ejercicio 10
# return {0: 16*psi, 1: 4*psi + 12*psi/n_gpu, 2: 2*psi + 14*psi/n_gpu, 3: 16*psi/n_gpu}[etapa]
# def burbuja(p, m): return (p - 1) / (m + p - 1)
# Ejercicio 11
# E_landauer = k_B * 300 * np.log(2);  E_flop_a100 = 400 / 312e12;  razon = E_flop_a100 / E_landauer